# Remoção de Ruído e Suavização de Imagens

Instituto Federal de Alagoas – Campus Arapiraca  
Processamento Digital de Sinais e Imagens  
**Grupo** – Jefferson Santos, José Davi e Gabriel Sousa

## 1. Preparação no Google Colab

Abra este notebook no Google Colab e selecione **Ambiente de execução → Executar tudo**. A primeira célula clona o repositório em `/content/pdsi-project` e entra nessa pasta. Após a execução, os painéis e o CSV ficam em `/content/pdsi-project/resultados/`. Você pode baixá-los pelo painel **Arquivos**.

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORIO = "https://github.com/jeffsdef/pdsi-project"
PASTA_PROJETO = Path("/content/pdsi-project")

if not (PASTA_PROJETO / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORIO, str(PASTA_PROJETO)], check=True)

os.chdir(PASTA_PROJETO)

In [ ]:
%pip install -q opencv-python-headless numpy matplotlib

In [ ]:
import csv

import cv2
import matplotlib.pyplot as plt
import numpy as np

PASTA_IMAGENS = Path("imagens")
PASTA_RESULTADOS = Path("resultados")
ARQUIVOS = ("araras.jpg", "casas.jpg", "estatua.jpg")

SEMENTE = 42
SIGMA_RUIDO = 20.0
PROPORCAO_IMPULSOS = 0.05
JANELA = 3

FILTROS = {
    "Média": lambda imagem: cv2.blur(imagem, (JANELA, JANELA)),
    "Mediana": lambda imagem: cv2.medianBlur(imagem, JANELA),
    "Gaussiano": lambda imagem: cv2.GaussianBlur(imagem, (JANELA, JANELA), sigmaX=0.8),
    "Bilateral": lambda imagem: cv2.bilateralFilter(imagem, d=JANELA, sigmaColor=60, sigmaSpace=2),
}

## 2. Simulação de ruído

In [ ]:
def carregar_imagem(caminho):
    dados = np.frombuffer(caminho.read_bytes(), dtype=np.uint8)
    imagem = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    if imagem is None:
        raise ValueError(f"Não foi possível decodificar {caminho}.")
    return imagem


def gerar_ruidos(imagem, rng):
    perturbacao = rng.normal(0, SIGMA_RUIDO, imagem.shape)
    gaussiano = np.clip(np.rint(imagem + perturbacao), 0, 255).astype(np.uint8)

    sorteio = rng.random(imagem.shape[:2])
    sal_e_pimenta = imagem.copy()
    sal_e_pimenta[sorteio < PROPORCAO_IMPULSOS / 2] = 0
    sal_e_pimenta[sorteio >= 1 - PROPORCAO_IMPULSOS / 2] = 255

    return {"Gaussiano": gaussiano, "Sal e pimenta": sal_e_pimenta}

## 3. Comparação visual e PSNR

In [ ]:
def calcular_psnr(original, processada):
    diferenca = original.astype(np.float64) - processada
    erro = np.mean(diferenca ** 2)
    return float("inf") if erro == 0 else 10 * np.log10(255**2 / erro)


def mostrar_painel(nome, ruido, original, comparacao, pontuacoes, caminho):
    itens = {"Original": original, **comparacao}
    figura, eixos = plt.subplots(1, len(itens), figsize=(12, 2.8), layout="constrained")
    figura.suptitle(f"{nome.capitalize()} | Ruído {ruido.lower()}")

    for eixo, (rotulo, imagem) in zip(eixos, itens.items()):
        eixo.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB), interpolation="nearest")
        eixo.set_axis_off()
        legenda = rotulo
        if rotulo in pontuacoes:
            legenda += f"\n{pontuacoes[rotulo]:.2f} dB"
        eixo.text(0.5, -0.04, legenda, transform=eixo.transAxes, ha="center", va="top")

    figura.savefig(caminho, dpi=160)
    plt.show()
    plt.close(figura)

## 4. Experimento e resultados preliminares

In [ ]:
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEMENTE)
linhas = []

for arquivo in ARQUIVOS:
    original = carregar_imagem(PASTA_IMAGENS / arquivo)
    nome = Path(arquivo).stem
    for ruido, ruidosa in gerar_ruidos(original, rng).items():
        comparacao = {"Sem filtro": ruidosa}
        for metodo, aplicar_filtro in FILTROS.items():
            comparacao[metodo] = aplicar_filtro(ruidosa)

        pontuacoes = {
            metodo: calcular_psnr(original, imagem)
            for metodo, imagem in comparacao.items()
        }
        for metodo, psnr in pontuacoes.items():
            linhas.append((arquivo, ruido, metodo, psnr))

        nome_ruido = ruido.lower().replace(" ", "_")
        mostrar_painel(
            nome, ruido, original, comparacao, pontuacoes,
            PASTA_RESULTADOS / f"{nome}_{nome_ruido}.png",
        )

with (PASTA_RESULTADOS / "psnr.csv").open("w", encoding="utf-8", newline="") as arquivo:
    tabela = csv.writer(arquivo)
    tabela.writerow(["imagem", "ruido", "metodo", "psnr_db"])
    tabela.writerows(linhas)

In [ ]:
for ruido in ("Gaussiano", "Sal e pimenta"):
    print(f"\nRuído {ruido.lower()} — PSNR médio")
    for metodo in ("Sem filtro", *FILTROS):
        valores = [
            psnr for _, tipo, filtro, psnr in linhas
            if tipo == ruido and filtro == metodo
        ]
        print(f"  {metodo:<12} {np.mean(valores):6.2f} dB")

[Repositório do projeto no GitHub](https://github.com/jeffsdef/pdsi-project)